In [7]:
from dotenv import load_dotenv
load_dotenv()
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [2]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = splitter.split_documents(docs)

In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore.from_documents(
    documents=split_docs, embedding=embeddings
)

In [ ]:
@tool
def retriever_tool(query: str):
    """
    This tool can help you the relevant data of the PDF Documents, and these documents have details about medical reports
    """
    docs = vectorstore.similarity_search(query=query, k=4)
    context = ""

    for doc in docs:
        context = doc.page_content + "\n\n"

    return context


In [15]:
llm = ChatOpenAI(model="gpt-5")

In [16]:
system_prompt = """
You are a helpful assistant that answers questions using the context provided. If you don't know the answer, just say that you don't know, don't try to make up an answer.
Always use the 'retiver_tool' to get the relevant context from the PDF documents.
"""

In [27]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=system_prompt
)

In [26]:
query = "What is the name of the patient and what is the name of the doctors?"
response = agent.invoke({"messages": [{"role": "user", "content": query}]})

In [29]:
result = response["messages"][-1].content

In [30]:
print(result)

- Patient: Ms. Nikita Chudhary
- Doctors:
  - Referring doctor: Dr. Nitin Nahar
  - Laboratory doctors: Dr. Swapnil Gupta; Dr. Kiran Bhargava Pathak
